# Blank Pulses

It is often desired to generate two pulses with an exact, known delay in between (during which no signals are played). Recall that the DMAs stream DAC signals by directly addressing the DAC sample memory, along with a "valid" signal that determines whether the memory output should be enabled or not. When no pulses are playing, this valid signal is low and the output of the memory is all zeroes. When a pulse is to be played, the DMA begins counting and sets the valid signal high; when it does this, the memory presents its contents at the address given by the DMA's counter. When the counter finishes, if no more pulses are to be played, the valid signal goes low and the memory returns to driving its output with all zeroes.

Because of the DMA's control over the memory output, we can emulate a delay by instructing the DMA to play a pulse as usual but to never assert the valid signal. This will cause the DMA counter to run as it would for any other pulse, but the memory output will remain as zeroes for that time. Then, when the counter completes, the next pulse can continue as usual, thereby creating a gap between the two pulses with no bubble cycles.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import time

from acadia.system import Acadia
from acadia.channel import Channel

No module named 'pyxrfdc'
No module named 'pyxrfclk'


In [2]:
acadia = Acadia()

capture_time = 5000e-9
pulse_time = 1000e-9
delay_time = 200e-9

pulse_channel = acadia.DAC(1)
capture_channel = acadia.ADC(1)

pulse_length = pulse_channel.seconds_to_samples(pulse_time)
capture_length = capture_channel.seconds_to_samples(capture_time)

pulse_memory = acadia.DACArray[pulse_channel.num](size=pulse_channel.seconds_to_bytes(pulse_time))
capture_memory = acadia.PLDDR0Array(size=capture_channel.seconds_to_bytes(capture_time))

# Create a sequence for the sequencer
@acadia.sequence
def sequence(a):
    with a.synchronizer():
        a.generate(pulse_channel, pulse_memory)
        a.generate_blank(pulse_channel, delay_time)
        a.generate(pulse_channel, pulse_memory)
        a.capture(capture_channel, capture_memory)

# Instruct the PS to create a 100ns cosine pulse and load it into memory
def program():   
    import numpy as np
    import time
    
    # Load the pulse into DAC memory
    pulse2 = (1/2) - (1/2)*np.cos(2*np.pi*np.arange(pulse_length)/pulse_length, dtype=np.complex64)
    pulse2_samples = pulse_channel.to_samples(pulse2)
    acadia.memcpy(pulse2_samples, pulse_memory)
    
    # Set up the channel properties
    pulse_channel.set_nyquist_zone(2)
    pulse_channel.configure_nco(frequency=1000e6)
    pulse_channel.set_vop(20000)
    capture_channel.set_nyquist_zone(2)
    capture_channel.set_dsa(0)
    
    # Clear the DDR array
    zeros = np.zeros(capture_length, dtype=np.complex64)
    zero_samples = capture_channel.to_samples(zeros)
    acadia.memcpy(zero_samples, capture_memory) 
    time.sleep(0.1) # Give the memory a moment to load
    
    # Configure the ADC switch
    acadia.configure()

    # Reset and run the sequencer
    acadia.sequencer_reset()
    acadia.sequencer_run(sequence)
    time.sleep(0.1)
    acadia.sequencer_halt()
    

In [ ]:
acadia.compile_all()
acadia.attach()
acadia.assemble(load=True)

program()

time_per_sample = capture_time / capture_channel.seconds_to_samples(capture_time)
trace = Channel.from_samples(capture_memory.memory)

times = np.arange(0, capture_time, time_per_sample)

fig,ax = plt.subplots()
ax.plot(times*1e6, np.real(trace), label="Re")
ax.plot(times*1e6, np.imag(trace), label="Im")
ax.set_xlabel("Time (us)")
ax.set_ylabel("Amplitude (\%FS)")
ax.grid()
ax.legend()
